In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display
import streamlit as st
import aquarel
aquarel.load_theme("minimal_light")

In [ ]:
cleanup = pd.read_csv('/Users/Ben.Pharris/Downloads/Channel Measurement by KPI - Media Team - <Channel_Owner>.csv') # <> anonymized

In [3]:
keep_cols = ['Channel Owner',	'Channel', 'KPI', 'KPI Level',	'January',	'February',	'March', 'April', 'May', 'June', 'July', 'August',	'September']

cleanup = cleanup[keep_cols]

cleanup = cleanup[cleanup['KPI Level'] == 'Primary']

cleanup = cleanup.drop(columns='KPI Level', errors='ignore')

cleanup = cleanup.fillna(np.nan)


In [4]:
kpiq3 = pd.read_excel('/Users/Ben.Pharris/Downloads/Channel_KPIS_Chart_Build.xlsx')

kpiq3 = pd.concat([kpiq3, cleanup], ignore_index=True)

all_months = ["January", "February", "March", "April", "May", "June", 
              "July", "August", "September", "October", "November", "December"]
month_cols = [col for col in all_months if col in kpiq3.columns]

# --- 1. Check for rows with all 0 or NA in month columns ---
is_zero_or_na = (kpiq3[month_cols] == 0) | (kpiq3[month_cols].isna())
is_row_all_zero_or_na = is_zero_or_na.all(axis=1)

kpiq3 = kpiq3[~is_row_all_zero_or_na]

# safely drop the Direction column if it exists
kpiq3 = kpiq3.drop(columns='Direction', errors='ignore')

kpiq3_wide = kpiq3.melt(
    id_vars = ['Channel Owner', 'Channel', 'KPI'],
    var_name = 'Month',
    value_name= 'Value',
)

kpiq3_wide.to_csv('/Users/Ben.Pharris/Downloads/KPI_Dashboard.csv')

In [6]:
import os, re, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.gridspec import GridSpec

# ----------------------------
# CONFIG
# ----------------------------
months      = ["January","February","March","April","May","June","July","August","September","October","November","December"]
months_abbr = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

OUTDIR_INDIVIDUAL = "charts_individual"
OUTDIR_GROUPED    = "charts_grouped"
MAKE_GROUPED_SHEETS = True
DPI = 200
AGG = "sum"   # "sum" | "mean" | "first"

# ----------------------------
# DATA CLEANUP
# ----------------------------
df = kpiq3_wide.copy()
df = df[df["Month"].isin(months)].copy()
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
df = df.dropna(subset=["Value"])
df["Month"] = pd.Categorical(df["Month"], categories=months, ordered=True)
if "Channel Owner" not in df.columns:
    df["Channel Owner"] = ""

# ----------------------------
# HELPERS
# ----------------------------
def safename(s):
    return re.sub(r"[^A-Za-z0-9._ -]+", "_", str(s))

def human_fmt(x):
    """K/M suffix; >100 no decimals; 1–100 one decimal; <1 up to 3 decimals."""
    ax = abs(x)
    if ax >= 1_000_000:
        v, suf = x / 1_000_000.0, "M"
    elif ax >= 1_000:
        v, suf = x / 1_000.0, "K"
    else:
        v, suf = x, ""
    av = abs(v)
    if av >= 100:
        s = f"{v:.0f}{suf}"
    elif av >= 1:
        s = f"{v:.1f}{suf}"
    elif av > 0:
        s = f"{v:.3f}{suf}".rstrip("0").rstrip(".")
    else:
        s = f"0{suf}"
    return s

fmt_y = FuncFormatter(lambda x, pos: human_fmt(x))

def monthly_series(g: pd.DataFrame) -> pd.Series:
    g = g.copy()
    g["Month"] = pd.Categorical(g["Month"], categories=months, ordered=True)
    s = (
        g.groupby("Month", observed=True)["Value"]
         .agg(AGG)
         .reindex(months)
    )
    return s

def table_data_for_series(g: pd.DataFrame):
    s = monthly_series(g)
    row = [("" if pd.isna(v) else human_fmt(v)) for v in s.values]
    return [row], ["Value"], months_abbr

def minimal_axes(ax):
    """Keep only left/bottom spines and ticks; no grid; no axis titles."""
    ax.grid(False)
    ax.set_xlabel("")
    ax.set_ylabel("")
    # Show only left & bottom spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(True)
    # Ticks only on left/bottom
    ax.tick_params(axis="both", which="both", length=4)

# ----------------------------
# 1) INDIVIDUAL (KPI, Channel) PLOTS + TABLE
# ----------------------------
os.makedirs(OUTDIR_INDIVIDUAL, exist_ok=True)

for (kpi, channel), g in df.groupby(["KPI", "Channel"], sort=True):
    g = g[["Month","Value","Channel Owner","Channel","KPI"]]
    if g.empty:
        continue

    owner = g["Channel Owner"].iloc[0] if "Channel Owner" in g.columns else ""
    title = f"{owner} - {channel} ({kpi})".strip(" -()")

    # aggregated series
    s = monthly_series(g)

    fig = plt.figure(figsize=(9, 5.2), constrained_layout=False)
    gs = GridSpec(nrows=2, ncols=1, height_ratios=[3.0, 1.1], figure=fig)

    # ----- plot (use numeric x for robust tick control)
    ax = fig.add_subplot(gs[0, 0])
    x = np.arange(len(months))
    ax.plot(x, s.values, marker="o", linestyle="-")
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(months_abbr)
    ax.yaxis.set_major_formatter(fmt_y)
    minimal_axes(ax)

    # ----- table under plot
    tdata, rlabels, clabels = table_data_for_series(g)
    axt = fig.add_subplot(gs[1, 0])
    axt.axis("off")
    table = axt.table(
        cellText=tdata,
        rowLabels=rlabels,
        colLabels=clabels,
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.5, 1.2)  # widen cells to prevent overflow

    fname = f"{safename(channel)}__{safename(kpi)}.png"
    fig.tight_layout(rect=[0, 0, 1, 1])
    fig.savefig(os.path.join(OUTDIR_INDIVIDUAL, fname), dpi=DPI)
    plt.close(fig)

print(f"Saved individual charts to: {os.path.abspath(OUTDIR_INDIVIDUAL)}")

# ----------------------------
# 2) OPTIONAL: GROUPED BY CHANNEL (small multiples + KPI×Month table)
# ----------------------------
if MAKE_GROUPED_SHEETS:
    os.makedirs(OUTDIR_GROUPED, exist_ok=True)

    for channel, gC in df.groupby("Channel", sort=True):
        if gC.empty:
            continue

        owner = (
            gC["Channel Owner"].mode().iat[0]
            if "Channel Owner" in gC.columns and not gC["Channel Owner"].isna().all()
            else ""
        )
        kpis = sorted(gC["KPI"].unique())
        n = len(kpis)
        cols = 3 if n >= 3 else n
        rows = max(1, (n + cols - 1) // cols)

        fig = plt.figure(figsize=(6*cols, 3.2*rows + 2.6), constrained_layout=False)
        gs = GridSpec(nrows=rows + 1, ncols=cols, height_ratios=[*( [3.0]*rows ), 1.4], figure=fig)

        # KPI small multiples
        for i, k in enumerate(kpis):
            r, c = divmod(i, cols)
            ax = fig.add_subplot(gs[r, c])
            sK = monthly_series(gC[gC["KPI"] == k])
            x = np.arange(len(months))
            ax.plot(x, sK.values, marker="o", linestyle="-")
            ax.set_title(str(k))
            ax.set_xticks(x)
            ax.set_xticklabels(months_abbr)
            ax.yaxis.set_major_formatter(fmt_y)
            minimal_axes(ax)

        # hide unused cells
        total_slots = rows * cols
        if n < total_slots:
            for j in range(n, total_slots):
                r, c = divmod(j, cols)
                ax = fig.add_subplot(gs[r, c])
                ax.axis("off")

        # KPI × Month table (human formatting)
        pivot = (
            gC.pivot_table(index="KPI", columns="Month", values="Value", aggfunc=AGG)
              .reindex(index=kpis, columns=months)
        )
        display_vals = pivot.applymap(lambda v: "" if pd.isna(v) else human_fmt(v))

        axt = fig.add_subplot(gs[rows, :])
        axt.axis("off")
        table = axt.table(
            cellText=display_vals.values,
            rowLabels=[str(x) for x in display_vals.index],
            colLabels=months_abbr,
            loc="center"
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.4, 1.2)

        fig.suptitle(f"{owner} - {channel}", y=0.995, fontsize=14)

        fname = f"{safename(channel)}__ALL_KPIs.png"
        fig.tight_layout(rect=[0, 0, 1, 0.97])
        fig.savefig(os.path.join(OUTDIR_GROUPED, fname), dpi=DPI)
        plt.close(fig)

    print(f"Saved grouped channel sheets to: {os.path.abspath(OUTDIR_GROUPED)}")


Saved individual charts to: /Users/Ben.Pharris/Documents/project-dev/Ad Hoc/charts_individual


/var/folders/r8/f4vjftzx1s93lfzrk2s0hk9w0000gq/T/ipykernel_57020/3244807750.py:183: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  gC.pivot_table(index="KPI", columns="Month", values="Value", aggfunc=AGG)
/var/folders/r8/f4vjftzx1s93lfzrk2s0hk9w0000gq/T/ipykernel_57020/3244807750.py:186: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  display_vals = pivot.applymap(lambda v: "" if pd.isna(v) else human_fmt(v))
/var/folders/r8/f4vjftzx1s93lfzrk2s0hk9w0000gq/T/ipykernel_57020/3244807750.py:183: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  gC.pivot_table(index="KPI", columns="Month", values="Value", aggfunc=AGG)
/var/folders/r8/f4vj

Saved grouped channel sheets to: /Users/Ben.Pharris/Documents/project-dev/Ad Hoc/charts_grouped


/var/folders/r8/f4vjftzx1s93lfzrk2s0hk9w0000gq/T/ipykernel_57020/3244807750.py:183: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  gC.pivot_table(index="KPI", columns="Month", values="Value", aggfunc=AGG)
/var/folders/r8/f4vjftzx1s93lfzrk2s0hk9w0000gq/T/ipykernel_57020/3244807750.py:186: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  display_vals = pivot.applymap(lambda v: "" if pd.isna(v) else human_fmt(v))
